In [ ]:
class SKNetBlock(nn.Module):
    def __init__(self, in_channels, reduction=16, num_branches=2):
        super(SKNetBlock, self).__init__()
        self.num_branches = num_branches
        self.in_channels = in_channels
        self.reduced_channels = in_channels // reduction

        # Branches: different kernel sizes for multi-scale representation
        self.branches = nn.ModuleList([
            nn.Conv2d(in_channels, in_channels, kernel_size=3 + 2 * i, padding=1 + i, groups=in_channels)
            for i in range(num_branches)
        ])

        # Fusion layers
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(in_channels, self.reduced_channels, kernel_size=1, bias=False)
        self.relu = nn.ReLU()
        self.fc2 = nn.Conv2d(self.reduced_channels, in_channels * num_branches, kernel_size=1, bias=False)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Compute outputs for each branch
        branch_outputs = [branch(x) for branch in self.branches]
        branch_outputs = torch.stack(branch_outputs, dim=1)  # Shape: [B, num_branches, C, H, W]

        # Compute attention weights
        attention = self.global_pool(x)  # Global context
        attention = self.fc1(attention)
        attention = self.relu(attention)
        attention = self.fc2(attention)  # Shape: [B, num_branches * C, 1, 1]
        attention = attention.view(x.size(0), self.num_branches, self.in_channels, 1, 1)
        attention = self.softmax(attention)  # Normalize across branches

        # Fuse branch outputs
        out = (branch_outputs * attention).sum(dim=1)  # Weighted sum across branches
        return out

